# N4B — L2 biological nodes/context summary

Read-only reproducibility notebook for L2 biological node/context ingestions. This notebook deliberately stays separate from interaction-source notebooks and **does not promote canonical KG files**.

Covered tranches:

- lncRNA nodes and disease/regulation/interaction source decisions (`t_35dddc93`, reviewed by `t_e3e2a5a0`).
- RBP/RNA CLIP / RNA-protein feature-only staging (`t_89b3ddaf`, reviewed by `t_010bc1e4`).
- expression/coexpression/correlation feature-context contract (`t_c8f1dbc0`, reviewed by `t_a2820a4e`).
- HPA cellular_component/protein localization staged artifacts (`t_4bda37e9`, fixed/rebuilt/reviewed by `t_2001e8ac`, `t_51714eaf`, `t_41852a2b`).
- Complex Portal protein_complex membership (`t_9d94edb6`, reviewed by `t_c34d9545`).
- UniProt PTM site/features (`t_ef541e0e`, reviewed by `t_13a70788`).
- OpenTargets/Ensembl `gene_paralog_gene` (`t_e7603b95`, reviewed by `t_b19c67f8`).


## Safety contract

Default execution is read-only and lightweight:

- only reads small JSON/text reports and source code metadata from this repository;
- does **not** write under `/mnt/gcs`, `gs://jouvencekb/kg/v2`, `data/kg`, `edges/`, `nodes/`, or `evidence/`;
- does **not** rerun source downloads or builders;
- treats all covered tranches as `staged-only` or `feature/context only` unless a separate human review explicitly authorizes canonical promotion.

If you need to refresh counts, run the listed builder/validation commands in a separate staging task and update this notebook from the resulting reports. Do not add heavy rebuild cells here.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Hard safety flag for this notebook: no build/promotion cells should ignore it.
READ_ONLY = True
assert READ_ONLY, "This notebook is intentionally read-only."


In [ ]:
SUMMARY_ROWS = [{'tranche': 'lncRNA nodes + LncRNADisease candidates', 'claim_level': 'staged-only', 'source_access_license': 'GENCODE v48 lncRNA GTF is acceptable for staged internal node pilot with attribution; LncRNADisease v3 TSV is public but explicit redistribution/commercial license was not found, so disease evidence remains candidate/staged only; RNAcentral/Lnc2Cancer/LncRNA2Target deferred until exact files/terms resolve.', 'schema_semantics': 'GENCODE lncRNA transcripts as candidate lncRNA/transcript nodes; LncRNADisease rows are disease-association/regulation context candidates, not canonical disease edges unless license + disease mapping + schema policy pass.', 'scripts_tests': 'historical legacy .omoc/scripts/stage_lncrna_l2f.py (provenance only); current reruns need reviewed manage_db/artifacts script plus tests/test_stage_lncrna_l2f.py', 'staged_prefix': 'gs://jouvencekb/kg/staging/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622', 'counts_rejects': 'validation: 197,208 lncRNA transcript nodes; 37,576 lncRNA gene records; 6,598 human LncRNADisease candidate rows; 0 active edges; 6,598 needs-review/rejected rows; remote parity 8/8 files.', 'review_status': 'Accepted by review t_e3e2a5a0 via parent t_35dddc93 as accept_staged_only.', 'canonical_gate': 'No canonical promotion. Requires explicit LncRNADisease license review, disease mapping policy, and lncRNA relation/schema approval.'}, {'tranche': 'RBP/RNA CLIP / RNA-protein', 'claim_level': 'feature/context only', 'source_access_license': 'ENCORI/starBase API/reference download is represented as CC-BY-4.0 per script source note; POSTAR3/other CLIP sources require their exact source terms before promotion.', 'schema_semantics': 'Direct RBP/RNA CLIP rows are preserved as candidates/features. The script refuses KG edges unless endpoints are source-native transcript IDs and protein IDs; current ENCORI rows expose RBP symbols, target gene names/IDs, and genomic coordinates, so no canonical transcript_interacts_protein edges are asserted.', 'scripts_tests': 'manage_db/build_staged_rbp_rna_interactions.py; tests/test_build_staged_rbp_rna_interactions.py', 'staged_prefix': 'gs://jouvencekb/kg/v2/staging/rbp-rna-clip-encori-pilot-20260622T135045Z/', 'counts_rejects': 'Parent/review status: no-edge/feature-only staging; canonical edge count is intentionally zero because source-native transcript/protein endpoints and lncRNA schema support are missing.', 'review_status': 'Accepted by review t_010bc1e4 via parent t_89b3ddaf as accept_feature_only.', 'canonical_gate': 'No canonical promotion. Requires endpoint mapping policy/schema prerequisites; do not infer coordinate→transcript or RBP-symbol→protein projections in this tranche.'}, {'tranche': 'Expression/coexpression/correlation feature context', 'claim_level': 'feature/context only', 'source_access_license': 'Contract only in this tranche; real source pilots are deferred until raw rows, licenses, endpoint mapping, and statistical context are approved.', 'schema_semantics': 'Non-causal feature/context tables outside edges/ and evidence/. Allowed evidence types are correlative, non_causal, predictive, association_score, or candidate_context; causal/mechanistic/regulatory/physical_interaction labels are explicitly forbidden.', 'scripts_tests': 'manage_db/kg_feature_context.py; tests/test_kg_feature_context.py', 'staged_prefix': 'No source prefix; this is a table contract, not a staged source build.', 'counts_rejects': 'Contract-level validation only; no active causal/mechanistic relation rows claimed.', 'review_status': 'Accepted by review t_a2820a4e via parent t_c8f1dbc0 as accept_feature_context_contract.', 'canonical_gate': 'No edge promotion. Build source-specific feature tables only after source package approval and non-causal statistical context review.'}, {'tranche': 'HPA cellular_component + protein localization', 'claim_level': 'staged-only', 'source_access_license': 'HPA proteinatlas.tsv.zip release HPA 25.1; GO basic releases/2026-05-19 and UniProt subcell current used for mapping. HPA attribution/terms must be rechecked before canonical promotion.', 'schema_semantics': 'cellular_component nodes from GO CC where feasible plus HPA local fallback IDs; protein_located_in_cellular_component edges from direct HPA UniProt localization labels resolved to existing protein nodes; optional cellular_component_subtype_of_cellular_component hierarchy; no gene→protein projection.', 'scripts_tests': 'manage_db/build_hpa_cellular_components.py; tests/test_build_hpa_cellular_components.py', 'staged_prefix': 'gs://jouvencekb/kg/staging/hpa-cellular-components-2026-06-22-rebuild-t_51714eaf/', 'counts_rejects': 'validation: 60 component nodes; 40 GO-mapped nodes; 18 HPA-local nodes; 276,356 protein-location edges; 536,967 evidence rows; 14 hierarchy edges/evidence; 0 endpoint/evidence anti-join misses; 291 HPA rows without protein mapping; all-ENSP expansion ambiguity documented.', 'review_status': 'Accepted by downstream review t_41852a2b after rebuild t_51714eaf via parent t_4bda37e9 as accept_staged_only.', 'canonical_gate': 'No canonical promotion. Requires explicit all-ENSP vs canonical-ENSP policy approval.'}, {'tranche': 'Complex Portal protein_complex membership', 'claim_level': 'staged-only', 'source_access_license': 'Complex Portal/EMBL-EBI release-pinned human complextab 9606.tsv, release 2026-01-14; license/terms should be reviewed before canonical promotion.', 'schema_semantics': 'protein_complex nodes preserve Complex Portal IDs; protein_part_of_protein_complex edges only for unique UniProt participants mapped to existing protein nodes; protein_complex_part_of_protein_complex edges only for explicit CPX-* nested participant tokens; evidence/reject reports preserve source tokens.', 'scripts_tests': 'manage_db/build_complex_portal_protein_complexes.py; tests/test_build_complex_portal_protein_complexes.py', 'staged_prefix': 'gs://jouvencekb/kg/staging/complex-portal-protein-complexes-2026-06-22/', 'counts_rejects': 'build report: 2,498 protein_complex nodes; 625 protein membership edges/evidence rows; 117 nested-complex edges/evidence rows; 9,359 rejected participants including 8,299 UniProt IDs mapping to multiple protein nodes, 207 unmapped UniProt IDs, and 853 unsupported namespaces; evidence support checks passed.', 'review_status': 'Accepted by review t_c34d9545 via parent t_9d94edb6 as accept_staged_only.', 'canonical_gate': 'No canonical promotion. Requires relation naming/schema approval and UniProt isoform/multi-mapping policy.'}, {'tranche': 'UniProt PTM sites/features', 'claim_level': 'staged-only', 'source_access_license': 'UniProtKB reviewed human PTM features; release-pinned download intended. BioGRID PTM xref verification exists separately for source-native expansion staging but disease/phenotype links are diagnostic-only here.', 'schema_semantics': 'ptm_site nodes plus protein_has_ptm_site edges/evidence. PTM feature types include Modified residue, Lipidation, Glycosylation, and Disulfide bond. Disease/phenotype comments are not inferred into KG edges; diagnostics table only.', 'scripts_tests': 'manage_db/build_uniprot_ptm_sites.py; tests/test_build_uniprot_ptm_sites.py; tests/test_biogrid_categorized_stage.py for BioGRID PTM xref staging context', 'staged_prefix': 'gs://jouvencekb/kg/staging/uniprot-ptm-sites-2026-06-22/', 'counts_rejects': 'Parent/review status: endpoint/evidence validation passed for staged ptm_site + protein_has_ptm_site. Local BioGRID PTM xref upload verification: 11/11 files present remotely, 0 missing, 0 extra. Exact UniProt PTM row counts should be read from the staged summary when the remote prefix is mounted/copied.', 'review_status': 'Accepted by review t_13a70788 via parent t_ef541e0e as accept_staged_only.', 'canonical_gate': 'No canonical promotion. PSI-MOD normalization, isoform handling, and any disease/phenotype relations require future schema/review gates.'}, {'tranche': 'OpenTargets/Ensembl gene_paralog_gene', 'claim_level': 'staged-only', 'source_access_license': 'OpenTargets target.homologues release 26.03; preserves source homology metadata and OpenTargets source-order semantics.', 'schema_semantics': 'human→human gene_paralog_gene directed source-order staged edges/evidence; mostly-null OT high-confidence is not treated as ortholog-like confidence; reverse/undirected handling must be an export-layer derived choice, not a source mutation.', 'scripts_tests': 'manage_db/build_opentargets_gene_paralogs.py; historical targeted report from legacy .omoc/staging/opentargets_gene_paralogs/reports/gene_paralog_gene_report.json (provenance only)', 'staged_prefix': 'gs://jouvencekb/kg/staging/source-native-expansion/gene_paralog_gene/opentargets-target-homologues-20260622/', 'counts_rejects': 'report: 78,691 OpenTargets source rows; 3,817,403 homologue records observed; 3,552,225 accepted human paralog records before endpoint filter; 3,544,825 source-order edges and evidence rows; 1,772,804 unique unordered pairs; 7,400 noncanonical gene endpoint rejects; 265,178 non-paralog homology-type rejects; endpoint/evidence checks passed.', 'review_status': 'Accepted by review t_b19c67f8 via parent t_e7603b95 as accept_staged_only.', 'canonical_gate': 'No canonical promotion. Requires human approval for canonical schema/KG/evidence-helper promotion and export semantics for mirrored/source-order rows.'}]
REPORT_PATHS = {
    'lncrna_validation': 'artifacts/reports/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622/reports/validation.json',
    'lncrna_source_audit': 'artifacts/reports/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622/source_audit.json',
    'lncrna_remote_parity': 'artifacts/reports/lncrna-l2f-remote-parity.json',
    'hpa_validation': 'artifacts/reports/hpa-cellular-components-2026-06-22-rebuild-t_51714eaf/validation.json',
    'hpa_manifest': 'artifacts/reports/hpa-cellular-components-2026-06-22-rebuild-t_51714eaf/manifest.json',
    'complex_portal_report': 'artifacts/reports/complex-portal-protein-complexes-2026-06-22/validation/complex_portal_build_report.json',
    'gene_paralog_report': 'artifacts/reports/opentargets_gene_paralogs/reports/gene_paralog_gene_report.json',
    'biogrid_ptm_xref_remote_verification': 'artifacts/reports/biogrid-ptm-xref-upload-verification.json',
}
HISTORICAL_LEGACY_REPORT_PATHS = {
    'note': 'Historical provenance only; do not use these legacy .omoc paths as current report inputs.',
    'legacy_examples': ['legacy .omoc/staging/...', 'legacy .omoc/reports/...'],
}
summary = pd.DataFrame(SUMMARY_ROWS)
summary[["tranche", "claim_level", "review_status", "canonical_gate"]]


## Consolidated status table

Interpretation:

- `staged-only`: artifacts may exist in staging prefixes and passed targeted review, but canonical KG promotion is not authorized.
- `feature/context only`: data/contract is useful to downstream models as non-causal context, but must not be represented as active causal/mechanistic KG edges.
- `canonical-ready`: none of the tranches below is claimed canonical-ready by this notebook.


In [ ]:
status_cols = [
    "tranche",
    "claim_level",
    "staged_prefix",
    "counts_rejects",
    "review_status",
    "canonical_gate",
]
summary[status_cols]


## Source/access/license and schema semantics

This table is the compact human-review surface: it records the source/access/license decision, what the schema means, and the command/test handles to reproduce the tranche in a staging-only task.


In [ ]:
semantic_cols = [
    "tranche",
    "source_access_license",
    "schema_semantics",
    "scripts_tests",
]
summary[semantic_cols]


## Local report inventory

The following helper reads only small report files if they are present locally. Missing reports are expected for remote-only prefixes; the notebook reports missing paths rather than downloading or rebuilding.


In [ ]:
def load_json_report(relative_path: str):
    path = REPO_ROOT / relative_path
    if not path.exists():
        return {"path": relative_path, "exists": False}
    with path.open() as handle:
        return {"path": relative_path, "exists": True, "data": json.load(handle)}

report_inventory = []
for name, relative_path in REPORT_PATHS.items():
    path = REPO_ROOT / relative_path
    report_inventory.append({
        "name": name,
        "path": relative_path,
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() else None,
    })
pd.DataFrame(report_inventory)


## Lightweight validation/readback summaries

These cells extract only count/check fields from local JSON reports. They are intentionally tolerant of missing files so the notebook remains safe on machines without GCS/FUSE or copied staging artifacts.


In [ ]:
def get_nested(data, *keys, default=None):
    cur = data
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

records = []

ln = load_json_report(REPORT_PATHS["lncrna_validation"])
if ln["exists"]:
    d = ln["data"]
    records.append({
        "tranche": "lncRNA",
        "local_report": ln["path"],
        "ok": d.get("ok"),
        "primary_counts": get_nested(d, "node_counts"),
        "reject_or_gate_counts": get_nested(d, "relation_counts"),
        "checks": {"evidence_support": d.get("evidence_support"), "endpoint_validation": d.get("endpoint_validation")},
    })

hpa = load_json_report(REPORT_PATHS["hpa_validation"])
if hpa["exists"]:
    d = hpa["data"]
    records.append({
        "tranche": "HPA cellular_component",
        "local_report": hpa["path"],
        "ok": True,
        "primary_counts": d.get("counts"),
        "reject_or_gate_counts": d.get("source_rows"),
        "checks": d.get("checks"),
    })

cp = load_json_report(REPORT_PATHS["complex_portal_report"])
if cp["exists"]:
    d = cp["data"]
    records.append({
        "tranche": "Complex Portal",
        "local_report": cp["path"],
        "ok": d.get("ok"),
        "primary_counts": d.get("counts"),
        "reject_or_gate_counts": d.get("source_counts"),
        "checks": d.get("checks"),
    })

gp = load_json_report(REPORT_PATHS["gene_paralog_report"])
if gp["exists"]:
    d = gp["data"]
    records.append({
        "tranche": "gene_paralog_gene",
        "local_report": gp["path"],
        "ok": True,
        "primary_counts": d.get("counts"),
        "reject_or_gate_counts": d.get("rejected_counts"),
        "checks": d.get("validation"),
    })

ptm = load_json_report(REPORT_PATHS["biogrid_ptm_xref_remote_verification"])
if ptm["exists"]:
    d = ptm["data"]
    records.append({
        "tranche": "UniProt/BioGRID PTM context",
        "local_report": ptm["path"],
        "ok": not d.get("missing") and d.get("remote_file_count") == d.get("local_file_count"),
        "primary_counts": {"remote_file_count": d.get("remote_file_count"), "local_file_count": d.get("local_file_count")},
        "reject_or_gate_counts": {"missing": d.get("missing"), "extra_remote": d.get("extra_remote")},
        "checks": {"remote_prefix": d.get("remote_prefix")},
    })

pd.DataFrame(records)


## Tranche notes

### lncRNA nodes and LncRNADisease candidate context

- Claim: `staged-only`.
- GENCODE v48 lncRNA transcript nodes are plausible staged inputs.
- LncRNADisease v3 rows are candidate disease-association/regulation context only; active edge rows are intentionally zero unless review gates are passed.
- Canonical gate: explicit LncRNADisease license review, disease ID mapping policy, and lncRNA schema/relation approval.

### RBP/RNA CLIP and RNA-protein features

- Claim: `feature/context only`.
- ENCORI/starBase CLIP rows expose useful RBP/RNA context but not source-native KG endpoint pairs sufficient for canonical transcript-protein edges.
- Do not silently project coordinates to transcripts or symbols to proteins.
- Canonical gate: endpoint mapping policy + schema support for the relevant RNA/protein relation.

### Expression/coexpression/correlation context

- Claim: `feature/context only`.
- These records live outside `edges/` and `evidence/`; they are non-causal model/context features.
- Forbidden semantics include causal, mechanistic, regulatory, direct binding, and PPI-like labels.
- Canonical gate: none for edges; future source-specific feature tables need their own source/license/statistical review.

### HPA cellular_component and protein localization

- Claim: `staged-only`.
- Direct HPA UniProt localization labels are mapped to existing protein nodes; no RNA/gene to protein projection.
- The accepted rebuild records HPA/GO/UniProt mapping releases and documents all-ENSP expansion ambiguity.
- Canonical gate: explicit approval of all-ENSP versus canonical-ENSP policy.

### Complex Portal protein_complex membership

- Claim: `staged-only`.
- Complex Portal IDs are retained as `protein_complex` staged node IDs; nested complex edges only come from explicit CPX participant tokens.
- Conservative unique-UniProt mapping rejects many participants; this is a policy choice, not a validation failure.
- Canonical gate: relation naming/schema approval and UniProt isoform/multi-mapping policy.

### UniProt PTM site/features

- Claim: `staged-only`.
- PTM sites and `protein_has_ptm_site` are accepted only as staged artifacts.
- Disease/phenotype mentions remain diagnostics only; no disease/phenotype edge is approved here.
- Canonical gate: PSI-MOD/feature normalization, isoform handling, and disease/phenotype schema review.

### OpenTargets/Ensembl gene_paralog_gene

- Claim: `staged-only`.
- Edges preserve OpenTargets source-order human→human paralog rows; export may later derive reverse/undirected forms, but staging does not mutate the source semantics.
- Mostly-null OpenTargets high-confidence values should not be interpreted as ortholog-style confidence.
- Canonical gate: human approval for relation/schema/evidence-helper promotion and explicit export semantics.


## Promotion gate checklist

Before any covered tranche can move from staged/context to canonical KG:

1. Source access/license is explicitly acceptable for the intended use and redistribution.
2. Endpoint types match source-native assertions; no RNA/gene/protein projection is introduced without policy approval.
3. Relation names and node types are registered/reviewed in schema docs/code.
4. Edges are supported by evidence rows and endpoint anti-joins pass against the canonical KG root.
5. Rejections/diagnostics are reviewed and not silently dropped.
6. A reviewer explicitly changes the status from `staged-only` or `feature/context only` to canonical-ready.

Current notebook conclusion: **none of the covered tranches is canonical-ready**.


In [ ]:
# Notebook self-check: do not accidentally label any covered tranche canonical-ready.
assert set(summary["claim_level"]) <= {"staged-only", "feature/context only"}
assert not summary["canonical_gate"].str.contains("authorized", case=False, regex=False).any()
summary["claim_level"].value_counts()
